## Summarization MiddleWare

The SummarizationMiddleware manages long-form student interactions by compressing aging dialogue into concise snapshots once a token threshold is reached. This process ensures the agent retains critical context—such as previously discussed course interests or fee quotes—without hitting the model's maximum limit.

### Key Benefits:

    Token Efficiency: Prevents "context overflow" in long-running enrollment consultations.

    Memory Retention: Preserves the last "n" messages for immediate accuracy while archiving the "big picture."

    Performance: Maintains low latency by preventing the input prompt from becoming excessively bloated.

### Problem 

Traditional training institute front-desk operations often struggle with high-volume inquiries, leading to fragmented student records and significant context loss during lengthy financial or course consultations. 

    Manual tracking of seat availability and fee installments is prone to error and creates bottlenecks in the enrollment pipeline. Without an automated, memory-persistent system, recruiters lose the "thread" of complex, multi-turn conversations, resulting in a degraded student experience and missed conversion opportunities.



### Solution

To address this issue This implementation leverages LangChain with AWS Bedrock to automate course eligibility checks and fee calculations through a professional AI receptionist. 
    
    By integrating SummarizationMiddleware, the system maintains long-term conversational coherence while optimizing token usage across extended student interactions. 

    InMemorySaver ensures seamless session persistence, providing a scalable solution for accurate, real-time administrative support.

In [1]:
from langchain_core.tools import tool
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# --- Domain Tools ---

@tool
def check_course_availability(course_name: str):
    """Checks if seats are available for a specific training course."""
    # Mock data for a training institute
    courses = {
        "Data Science": "5 seats left (Batch starts Monday)",
        "Web Development": "Waitlist only",
        "AI Ethics": "12 seats available"
    }
    return courses.get(course_name, "Course not found. Please ask about Data Science or Web Dev.")

@tool
def calculate_installment_plan(total_fee: float, months: int):
    """Calculates the monthly payment for a student fee plan."""
    if months <= 0: 
        return "Duration must be at least 1 month."
    monthly = total_fee / months
    return f"The installment plan is {months} payments of ${monthly:.2f} per month."

tools = [check_course_availability, calculate_installment_plan]

In [3]:
# Main Agent Model 
model = ChatBedrockConverse(
    model_id="cohere.command-r-plus-v1:0",
    region_name="us-east-1", 
    temperature=0.3
)

# Summarization Model
summary_model = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

In [4]:
# System prompt to ground the agent in the receptionist persona
SYSTEM_PROMPT = """You are the professional receptionist for 'FutureTech Institute'. 
Your goal is to help prospective students with course info and fee queries. 
Be polite, helpful, and concise."""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            trigger=("tokens", 400), # Summarize when context hits 400 tokens
            keep=("messages", 6),     # Always keep the last 6 messages raw
        ),
    ],
)

In [7]:
from rich import print
config = {"configurable": {"thread_id": "student_001"}}

while True:
    query=input("User_Query:")
    if query=='exit':
        break
    response = agent.invoke(
        {"messages": [{"role": "user", "content":query }]},
        config=config
    )

    print(response["messages"][-1].content)

User_Query: Hi


Hello, how can I help?

User_Query: I want to check if any seats available for AI Ethics course?


There are 12 seats available for the AI Ethics course.

User_Query: Ok . What is the fees? how can I pay?? 


...

User_Query: No. Some one told the fees is arround 4000$. Can I pay in installments??


Yes, you can pay in instalments. If you would like to pay over 12 months, the monthly payment would be $333.33.

User_Query: Thank you


You're welcome.

User_Query: exit


In [8]:
print(response)

{
    'messages': [
        HumanMessage(
            content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal 
is to check the availability of seats and the fees for the AI Ethics course, and to understand the payment 
process.\n\n## SUMMARY\n- The user inquired about the availability of seats for the AI Ethics course and confirmed 
there are 12 seats available.\n- The user asked about the fees for the course and was informed that the fees are 
around $4000.\n- The user asked if they can pay in installments.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Inform 
the user about the fees for the AI Ethics course.\n- Explain the payment process for the course, including the 
option to pay in installments.",
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='69ce4ec4-19bf-40dc-b117-d81eddb334f6'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': "I will use the 'calculate_installment_plan' tool to calculate the monthly payment for 
an instalment plan for a fee of $4000."
                },
                {
                    'type': 'tool_use',
                    'name': 'calculate_installment_plan',
                    'input': {'months': None, 'total_fee': 4000},
                    'id': 'tooluse_TZx8arO0NygEWobpmjDimi'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '71982bb8-682f-4c00-bc90-5f2d087c77ca',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Wed, 25 Mar 2026 15:32:39 GMT',
                        'content-type': 'application/json',
                        'content-length': '465',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '71982bb8-682f-4c00-bc90-5f2d087c77ca'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [2624]},
                'model_provider': 'bedrock_converse',
                'model_name': 'cohere.command-r-plus-v1:0'
            },
            id='lc_run--019d25a0-5046-7020-8e83-7911bb418887-0',
            tool_calls=[
                {
                    'name': 'calculate_installment_plan',
                    'args': {'months': None, 'total_fee': 4000},
                    'id': 'tooluse_TZx8arO0NygEWobpmjDimi',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 265,
                'output_tokens': 52,
                'total_tokens': 317,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content="Error invoking tool 'calculate_installment_plan' with kwargs {'months': None, 'total_fee': 
4000} with error:\n months: Input should be a valid integer\n Please fix the error and try again.",
            name='calculate_installment_plan',
            id='e3587e35-7e86-4fbd-adec-337dde84086c',
            tool_call_id='tooluse_TZx8arO0NygEWobpmjDimi',
            status='error'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': 'The tool has returned an error message. I will now run the tool again, this time 
specifying the number of months for the instalment plan.'
                },
                {
                    'type': 'tool_use',
                    'name': 'calculate_installment_plan',
                    'input': {'months': 12, 'total_fee': 4000},
                    'id': 'tooluse_nVg6OGZUsoqLcPNTyTep05'
                }
            ],
            additional_kwargs={},
         